[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensioai/blog/blob/master/028_pytorch_nn/pytorch_nn.ipynb)

# Pytorch - Redes Neuronales

En el post [anterior](https://sensioai.com/blog/027_pytorch_intro) hicimos una introducción al framework de `redes neuronales` `Pytorch`. Hablamos de sus tres elementos fundamentales: el objeto `tensor` (similar al `array` de `NumPy`) `autograd` (que nos permite calcular derivadas de manera automáticas) y el soporte GPU. En este post vamos a entrar en detalle en la  funcionalidad que nos ofrece la librería para diseñar redes neuronales de manera flexible.

In [1]:
import torch

## Modelos secuenciales

La forma más sencilla de definir una `red neuronal` en `Pytorch` es utilizando la clase `Sequentail`. Esta clase nos permite definir una secuencia de capas, que se aplicarán de manera secuencial (las salidas de una capa serán la entrada de la siguiente). Ésto ya lo conocemos de posts anteriores, ya que es la forma ideal de definir un `Perceptrón Multicapa`.

In [31]:
D_in, H, D_out = 3072, 300, 7

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)

El modelo anterior es un `MLP` con 784 entradas, 100 neuronas en la capa oculta y 10 salidas. Podemos usar este modelo para hacer un clasificador de imágenes con el dataset MNIST. Pero primero, vamos a ver como podemos calcular las salidas del modelo a partir de unas entradas de ejemplo.

In [32]:
outputs = model(torch.randn(600, 3072))
outputs.shape   

torch.Size([600, 7])

In [33]:
print(outputs[0][:])

tensor([ 0.2345, -0.1106,  0.1309, -0.2281,  0.0210, -0.1911,  0.2262],
       grad_fn=<SliceBackward0>)


Como puedes ver, simplemente le pasamos los inputs al modelo (llamándolo como una función). En este caso, usamos un tensor con 64 vectores de 784 valores. Es importante remarcar que los modelos de `Pytorch` (por lo general) siempre esperan que la primera dimensión sea la dimensión *batch*. Si queremos entrenar esta red en una GPU, es tan sencillo como

In [34]:
model

Sequential(
  (0): Linear(in_features=3072, out_features=300, bias=True)
  (1): ReLU()
  (2): Linear(in_features=300, out_features=7, bias=True)
)

In [6]:
#model.to("cuda")

Vamos a ver ahora como entrenar este modelo con el dataset MNIST.

In [65]:
import numpy as np
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# 1. Redimensionar a 32x32 y convertir a tensores
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# 2. Cargar las imágenes de la carpeta Vehicles
dataset = ImageFolder(root="Vehicles", transform=transform)

# 3. Cargar todas las imágenes en memoria para extraer X e Y
loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
images, labels = next(iter(loader))

# Aplanamos cada imagen a un vector de 3072 entradas (3 * 32 * 32)
X = images.view(images.shape[0], -1).numpy()
Y = labels.numpy()

X.shape, Y.shape

((5589, 3072), (5589,))

In [36]:
import numpy as np
from sklearn.model_selection import train_test_split

x_2 = np.array(X)
y_2 = np.array(Y)

# División estratificada (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(
    x_2, y_2, test_size=0.2, random_state=42, stratify=y_2
)

# Convertir tipos de datos
y_train = y_train.astype(np.int64)
y_test = y_test.astype(np.int64)

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")


#X_train, X_test, y_train, y_test = X[:60000] / 255., X[60000:] / 255., Y[:60000].astype(np.float32), Y[60000:].astype(np.float32)

X_train: (4471, 3072) | y_train: (4471,)
X_test:  (1118, 3072)  | y_test:  (1118,)


In [37]:
# función de pérdida y derivada

def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1,keepdims=True)

def cross_entropy(output, target):
    logits = output[torch.arange(len(output)), target]
    loss = - logits + torch.log(torch.sum(torch.exp(output), axis=-1))
    loss = loss.mean()
    return loss

In [38]:
X_train

array([[0.08627451, 0.08627451, 0.08235294, ..., 0.92156863, 0.93333334,
        0.9411765 ],
       [0.93333334, 0.93333334, 0.93333334, ..., 0.6627451 , 0.65882355,
        0.65882355],
       [0.6627451 , 0.6313726 , 0.65882355, ..., 0.8039216 , 0.81960785,
        0.827451  ],
       ...,
       [0.91764706, 0.9254902 , 0.93333334, ..., 0.3529412 , 0.35686275,
        0.3372549 ],
       [0.47058824, 0.48235294, 0.4862745 , ..., 0.6117647 , 0.6156863 ,
        0.60784316],
       [1.        , 1.        , 1.        , ..., 0.99607843, 0.9882353 ,
        0.9843137 ]], shape=(4471, 3072), dtype=float32)

In [39]:
torch.cuda.is_available()

False

In [40]:
print(X)

[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.59607846 0.5764706  0.54901963 ... 0.6392157  0.654902   0.6392157 ]
 [0.10980392 0.10196079 0.09803922 ... 0.14117648 0.12156863 0.11372549]
 ...
 [0.7411765  0.7411765  0.7294118  ... 0.7764706  0.76862746 0.7607843 ]
 [0.9647059  0.92156863 0.85490197 ... 0.47843137 0.50980395 0.5764706 ]
 [0.41568628 0.47843137 0.53333336 ... 0.32156864 0.49411765 0.5254902 ]]


In [41]:
# convertimos datos a tensores para trabajar en CPU

X_t = torch.from_numpy(X_train).float()
Y_t = torch.from_numpy(y_train).long()

# aseguramos el modelo en CPU
model = model.cpu()

# bucle entrenamiento
epochs = 350
lr = 0.05
log_each = 10
l = []

for e in range(1, epochs + 1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = cross_entropy(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    model.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    with torch.no_grad():
        for param in model.parameters():
            param -= lr * param.grad

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

Epoch 10/350 Loss 1.87551
Epoch 20/350 Loss 1.81792
Epoch 30/350 Loss 1.77220
Epoch 40/350 Loss 1.75599
Epoch 50/350 Loss 1.72669
Epoch 60/350 Loss 1.70282
Epoch 70/350 Loss 1.67809
Epoch 80/350 Loss 1.66200
Epoch 90/350 Loss 1.64539
Epoch 100/350 Loss 1.62856
Epoch 110/350 Loss 1.61376
Epoch 120/350 Loss 1.60093
Epoch 130/350 Loss 1.58643
Epoch 140/350 Loss 1.57516
Epoch 150/350 Loss 1.56422
Epoch 160/350 Loss 1.55133
Epoch 170/350 Loss 1.54083
Epoch 180/350 Loss 1.53103
Epoch 190/350 Loss 1.52062
Epoch 200/350 Loss 1.51223
Epoch 210/350 Loss 1.50250
Epoch 220/350 Loss 1.49412
Epoch 230/350 Loss 1.48537
Epoch 240/350 Loss 1.47749
Epoch 250/350 Loss 1.46983
Epoch 260/350 Loss 1.46204
Epoch 270/350 Loss 1.45472
Epoch 280/350 Loss 1.44745
Epoch 290/350 Loss 1.44070
Epoch 300/350 Loss 1.43405
Epoch 310/350 Loss 1.42781
Epoch 320/350 Loss 1.42124
Epoch 330/350 Loss 1.41485
Epoch 340/350 Loss 1.40912
Epoch 350/350 Loss 1.40287


Como puedes observar en el ejemplo, podemos calcular la salida del modelo con una simple línea. Luego calculamos la función de pérdida, y llamando a la función `backward` `Pytorch` se encarga de calcular las derivadas de la misma con respecto a todos los parámetros del modelo automáticamente (si no queremos acumular estos gradientes, nos aseguramos de llamar a la función `zero_grad` para ponerlos a cero antes de calcularlos). Por útlimo, podemos iterar por los parámetros del modelo aplicando la regla de actualización deseada (en este caso usamos `descenso por gradiente`).

In [42]:
from sklearn.metrics import accuracy_score

def evaluate(x):
    model.eval()
    with torch.no_grad():
        y_pred = model(x)
        y_probas = softmax(y_pred)
        return torch.argmax(y_probas, axis=1)

# Sin .cuda() porque usas CPU
y_pred = evaluate(torch.from_numpy(X_test).float())

# Exactitud final
acc = accuracy_score(y_test, y_pred.numpy())
print(f"Accuracy en Test: {acc * 100:.2f}%")

Accuracy en Test: 53.49%


In [43]:
from sklearn.metrics import classification_report, confusion_matrix

print("Reporte detallado por clase:")
print(classification_report(y_test, y_pred.numpy(), digits=4))

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred.numpy()))

Reporte detallado por clase:
              precision    recall  f1-score   support

           0     0.5086    0.5563    0.5313       160
           1     0.6833    0.7688    0.7235       160
           2     0.5763    0.2152    0.3134       158
           3     0.4600    0.5750    0.5111       160
           4     0.5274    0.6625    0.5873       160
           5     0.7436    0.3625    0.4874       160
           6     0.4267    0.6000    0.4987       160

    accuracy                         0.5349      1118
   macro avg     0.5608    0.5343    0.5218      1118
weighted avg     0.5608    0.5349    0.5222      1118

Matriz de confusión:
[[ 89  11   7  23  10   3  17]
 [  8 123   1   3   3   1  21]
 [ 17  11  34  45   8   4  39]
 [ 29   9   4  92   9   3  14]
 [  9   6   1  13 106   8  17]
 [ 13   7   3   5  53  58  21]
 [ 10  13   9  19  12   1  96]]


Existen algunos tipos de capas que se comportan diferente en función de si estamos entrenando la red o usándola para generar predicciones. Podemos controlar el modo en el que queremos que esté nuestra red con las funciones `train` y `eval`.

## Optimizadores y Funciones de pérdida

En el ejemplo anterior hemos calculado la función de pérdida y aplicado la regla de optimización de forma manual. Sin embargo, `Pytorch` nos ofrece funcionalidad que nos abstrae estos cálculos ofreciendo además flexibilidad para aplicar diferentes funciones de pérdida o algoritmos de optimización de manera sencilla. Podemos encontrar diferentes funciones de pérdida ya implementadas en el paquete `torch.nn`.

In [15]:
criterion = torch.nn.CrossEntropyLoss()

Mientras que los optimizadores se encuentran en el paquete `torch.optim`

In [16]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

Puedes ver la lista completa de funciones de pérdida y optimizadores disponibles en la [documentación](https://pytorch.org/docs/stable/index.html), aunque como ya has visto siempre puedes definir los tuyos propios fácilmente.

Una vez definidos estos dos objetos, nuestro bucle de entrenamiento se simplifica considerablemente.

In [17]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score

# 1. Dimensiones adaptadas a tus vehículos
D_in, H, D_out = 3072, 100, 7

# 2. Modelo creado directamente en CPU (sin .to("cuda"))
model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)

# 3. Datos a tensores en CPU
X_t = torch.from_numpy(X_train).float()
Y_t = torch.from_numpy(y_train).long()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# 4. Bucle de entrenamiento
epochs = 300
log_each = 20
l = []
model.train()

for e in range(1, epochs + 1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop
    loss.backward()

    # update de los pesos con optimizador SGD
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

# 5. Evaluación en conjunto de prueba (sin .cuda())
y_pred = evaluate(torch.from_numpy(X_test).float())
acc = accuracy_score(y_test, y_pred.cpu().numpy())
print(f"\nExactitud final (Accuracy): {acc * 100:.2f}%")

Epoch 20/300 Loss 1.83600
Epoch 40/300 Loss 1.77638
Epoch 60/300 Loss 1.72119
Epoch 80/300 Loss 1.68433
Epoch 100/300 Loss 1.64943
Epoch 120/300 Loss 1.62038
Epoch 140/300 Loss 1.59466
Epoch 160/300 Loss 1.57123
Epoch 180/300 Loss 1.54941
Epoch 200/300 Loss 1.52904
Epoch 220/300 Loss 1.51078
Epoch 240/300 Loss 1.49355
Epoch 260/300 Loss 1.47868
Epoch 280/300 Loss 1.46363
Epoch 300/300 Loss 1.45046

Exactitud final (Accuracy): 51.07%


## Modelos custom

Si bien en muchos casos definir una `red neuronal` como una secuencia de capas es suficiente, en otros casos será un factor limitante. Un ejemplo son las redes residuales, en las que no sólo utilizamos la salida de una capa para alimentar la siguiente si no que, además, le sumamos su propia entrada. Este tipo de arquitectura no puede ser definida con la clase `Sequential`, y para ello necesitamos hacer un modelo *customizado*. Para ello, `Pytroch` nos ofrece la siguiente sintaxis.

In [44]:
# creamos una clase que hereda de `torch.nn.Module`

class ModeloPersonalizado(torch.nn.Module):

    # constructor
    def __init__(self, D_in, H, D_out):

        # llamamos al constructor de la clase madre
        super(ModeloPersonalizado, self).__init__()

        # definimos nuestras capas
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    # lógica para calcular las salidas de la red
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

En primer lugar, necesitamos definir una nueva clase que herede de la clase `torch.nn.Module`. Esta clase madre aportará toda la funcionalidad esencial que necesita una `red neuronal` (soporte GPU, iterar por sus parámeteros, etc). Luego, en esta clase necesitamos definir mínimos dos funciones:

- `init`: en el constructor llamaremos al constructor de la clase madre y después definiremos todas las capas que querramos usar en la red.
- `forward`: en esta función definimos toda la lógica que aplicaremos desde que recibimos los inputs hasta que devolvemos los outputs.

En el ejemplo anterior simplemente hemos replicado la misma red (puedes conseguir el mismo efecto usando la clase `Sequential`).

In [45]:
# Instanciamos el modelo con las dimensiones de tus vehículos
model = ModeloPersonalizado(3072, 300, 7)

# Código para saber si el modelo está botando los datos en las cantidades correctas
x_prueba = torch.randn(500, 3072)
print(x_prueba)

outputs = model(x_prueba)
outputs.shape

tensor([[-1.0122,  1.1063,  0.3149,  ...,  0.5619,  0.2251,  0.5610],
        [-0.0665, -0.2742,  2.0588,  ..., -1.9485, -1.0933,  1.1008],
        [ 0.0357,  2.7644,  1.7326,  ...,  1.2363, -0.1015,  2.3318],
        ...,
        [ 0.0118, -0.1561, -0.7924,  ...,  1.2665, -1.6315,  1.3307],
        [ 0.2903, -1.5769,  2.3776,  ...,  0.4457,  0.7809,  0.8736],
        [-0.5921, -2.1360,  0.4309,  ..., -0.8675, -0.0473,  0.3174]])


torch.Size([500, 7])

Ahora, podemos entrenar nuestra red de la misma forma que lo hemos hecho anteriormente.

In [46]:
# Aseguramos el modelo en CPU (sin .to("cuda"))
model = model.cpu()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

epochs = 100
log_each = 10
l = []
model.train()

for e in range(1, epochs + 1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

# Evaluación en CPU (sin .cuda())
y_pred = evaluate(torch.from_numpy(X_test).float())
acc = accuracy_score(y_test, y_pred.cpu().numpy())
print(f"\nExactitud final (Accuracy): {acc * 100:.2f}%")

Epoch 10/100 Loss 1.88489
Epoch 20/100 Loss 1.82958
Epoch 30/100 Loss 1.78004
Epoch 40/100 Loss 1.76247
Epoch 50/100 Loss 1.73518
Epoch 60/100 Loss 1.70923
Epoch 70/100 Loss 1.68622
Epoch 80/100 Loss 1.67006
Epoch 90/100 Loss 1.65162
Epoch 100/100 Loss 1.63562

Exactitud final (Accuracy): 41.50%


Aquí puedes ver otro ejemplo de como definir un `MLP` con conexiones residuales, algo que no podemos hacer simplemente usando un modelo secuencial.

In [47]:
class ModelCustom2(torch.nn.Module):

    def __init__(self, D_in, H, D_out):
        super(ModelCustom2, self).__init__()
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    def forward(self, x):
        x1 = self.fc1(x)
        x = self.relu(x1)
        x = self.fc2(x + x1)
        return x

In [48]:
# 1. Instanciamos el modelo residual con 3072 entradas y 7 salidas en CPU
model = ModelCustom2(3072, 300, 7)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

epochs = 100
log_each = 10
l = []
model.train()

for e in range(1, epochs + 1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

# 2. Evaluación en Test directamente en CPU (sin .cuda())
y_pred = evaluate(torch.from_numpy(X_test).float())
acc = accuracy_score(y_test, y_pred.cpu().numpy())
print(f"\nExactitud final (Accuracy): {acc * 100:.2f}%")

Epoch 10/100 Loss 2.76985
Epoch 20/100 Loss 2.62879
Epoch 30/100 Loss 2.33203
Epoch 40/100 Loss 2.16276
Epoch 50/100 Loss 2.06868
Epoch 60/100 Loss 1.99166
Epoch 70/100 Loss 1.93910
Epoch 80/100 Loss 1.89125
Epoch 90/100 Loss 1.85700
Epoch 100/100 Loss 1.81953

Exactitud final (Accuracy): 43.74%


De esta manera, tenemos mucha flexibilidad para definir nuestras redes.

## Accediendo a las capas de una red

En ocasiones queremos acceder a una capa en particular de nuestra red. Para ello, podemos acceder utilizando su nombre.

In [49]:
model

ModelCustom2(
  (fc1): Linear(in_features=3072, out_features=300, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=300, out_features=7, bias=True)
)

In [50]:
model.fc1

Linear(in_features=3072, out_features=300, bias=True)

También podemos acceder directamente a los tensores que contienen los parámetros con las propiedades adecuadas

In [51]:
model.fc1.weight

Parameter containing:
tensor([[ 0.0052,  0.0005, -0.0061,  ...,  0.0139, -0.0070, -0.0093],
        [ 0.0052,  0.0166,  0.0167,  ..., -0.0162, -0.0145, -0.0005],
        [-0.0163, -0.0144,  0.0134,  ..., -0.0012,  0.0182, -0.0008],
        ...,
        [ 0.0193,  0.0190,  0.0174,  ..., -0.0142, -0.0110, -0.0034],
        [ 0.0108,  0.0110, -0.0016,  ..., -0.0148, -0.0080, -0.0013],
        [-0.0121, -0.0138,  0.0043,  ..., -0.0098,  0.0053,  0.0048]],
       requires_grad=True)

In [52]:
model.fc1.bias

Parameter containing:
tensor([-0.0134,  0.0052, -0.0152, -0.0130,  0.0012, -0.0137,  0.0104,  0.0059,
        -0.0107, -0.0138, -0.0126, -0.0032,  0.0008,  0.0062,  0.0073, -0.0120,
         0.0004,  0.0129, -0.0187,  0.0019,  0.0077,  0.0139, -0.0141,  0.0047,
         0.0123, -0.0163, -0.0072, -0.0128,  0.0161, -0.0183,  0.0081,  0.0083,
        -0.0159,  0.0032, -0.0014,  0.0118,  0.0231,  0.0142, -0.0081, -0.0130,
        -0.0054,  0.0209,  0.0049, -0.0024, -0.0091,  0.0070,  0.0204, -0.0047,
        -0.0042,  0.0130, -0.0103, -0.0076,  0.0082, -0.0076, -0.0113,  0.0015,
         0.0213,  0.0163, -0.0030, -0.0010,  0.0006, -0.0214, -0.0006, -0.0221,
        -0.0119, -0.0138,  0.0028, -0.0005, -0.0043,  0.0081,  0.0008, -0.0200,
        -0.0186,  0.0017,  0.0133,  0.0112,  0.0117,  0.0028, -0.0061, -0.0065,
         0.0022,  0.0109,  0.0085,  0.0131,  0.0063, -0.0060,  0.0083,  0.0066,
         0.0096,  0.0017,  0.0022,  0.0118,  0.0049, -0.0059,  0.0050, -0.0023,
        -0.0135,  

Es posible sobreescribir una capa de la siguiente manera

In [55]:
model.fc2 = torch.nn.Linear(100, 7)

model

ModelCustom2(
  (fc1): Linear(in_features=3072, out_features=300, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=7, bias=True)
)

Ahora, la capa final de nuestra red tiene solo una salida. Esta nueva capa ha sido inicializada de manera aleatoria, por lo que esta nueva red no nos va a servir de mucho. Sin embargo, podríamos volver a entrenar esta red en otro problema en el que solo necesitemos una salida aprovechando los pesos que ya hemos entrenado anteriormente con el dataset MNIST. Esto es la base del *transfer learning*, una técnica que utilizaremos muchísimo más adelante y la cual explicaremos en detalle.

A continuación encontrarás varios trucos a la hora de crear redes neuronales a partir de otras que te pueden resultar útiles.

In [56]:
# obtener una lista con las capas de una red

list(model.children())

[Linear(in_features=3072, out_features=300, bias=True),
 ReLU(),
 Linear(in_features=100, out_features=7, bias=True)]

In [57]:
# crear nueva red a partir de la lista (excluyendo las útlimas dos capa)

new_model = torch.nn.Sequential(*list(model.children())[:-2])
new_model

Sequential(
  (0): Linear(in_features=3072, out_features=300, bias=True)
)

In [58]:
# crear nueva red a partir de la lista (excluyendo las útlima capa)

new_model = torch.nn.ModuleList(list(model.children())[:-1])
new_model

ModuleList(
  (0): Linear(in_features=3072, out_features=300, bias=True)
  (1): ReLU()
)

## Resumen

En este post hemos visto la funcionalidad que `Pytorch` nos ofrece a la hora de definir y entrenar nuestras `redes neuronales`. El paquete `torch.nn` contiene todo lo necesario para diseñar nuestros modelos, ya sea de manera secuencial o con una clase *custom* para arquitecturas más complicadas. También nos da muchas funciones de pérdida que podemos usar directamente para entrenar las redes. Te recomiendo encarecidamente que le eches un vistazo a la [documentación](https://pytorch.org/docs/stable/nn.html) par hacerte una idea de todo lo que puedes hacer. También hemos visto como el paquete `torch.optim` nos oferece algoritmos de optimización que también nos hacen la vida más fácil a la hora de entrenar nuestras redes.